In [6]:
import sqlite3
import pandas as pd

# Load clean data
ipl = pd.read_csv('E:\\ipl_dataset1\\ipl_clean.csv', 
                   low_memory=False)

# Create SQLite database
conn = sqlite3.connect('E:\\ipl_dataset1\\ipl.db')
ipl.to_sql('ipl', conn, if_exists='replace', index=False)
print("Database created!")

Database created!


In [7]:
# Checking if table exists and has data
query = """
SELECT COUNT(*) as total_rows FROM ipl
"""
result = pd.read_sql(query, conn)
print(result)

   total_rows
0      278205


In [17]:
top_run_scorers = """
SELECT batter,
       SUM(runs_batter) as total_runs
FROM ipl
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 10
"""

df_top_scorers = pd.read_sql(top_run_scorers, conn)
print(df_top_scorers)

           batter  total_runs
0         V Kohli        8671
1       RG Sharma        7048
2        S Dhawan        6769
3       DA Warner        6567
4        SK Raina        5536
5        MS Dhoni        5439
6        KL Rahul        5235
7  AB de Villiers        5181
8       AM Rahane        5032
9        CH Gayle        4997


In [29]:
team_win_rates = """
WITH wins AS (
    SELECT 
        CASE 
            WHEN match_won_by = 'Delhi Daredevils' 
                THEN 'Delhi Capitals'
            WHEN match_won_by = 'Royal Challengers Bangalore' 
                THEN 'Royal Challengers Bengaluru'
            WHEN match_won_by = 'Kings XI Punjab' 
                THEN 'Punjab Kings'
            WHEN match_won_by = 'Deccan Chargers' 
                THEN 'Sunrisers Hyderabad'
            WHEN match_won_by = 'Rising Pune Supergiant' 
                THEN 'Rising Pune Supergiants'
            ELSE match_won_by 
        END as team,
        COUNT(DISTINCT match_id) as total_wins
    FROM ipl
    WHERE match_won_by != 'Unknown'
    GROUP BY team
),
total AS (
    SELECT batting_team as team,
           COUNT(DISTINCT match_id) as total_matches
    FROM ipl
    GROUP BY batting_team
)
SELECT w.team,
       w.total_wins,
       t.total_matches,
       ROUND(w.total_wins * 100.0 / 
             t.total_matches, 2) as win_rate
FROM wins w
JOIN total t ON w.team = t.team
ORDER BY win_rate DESC
"""

df_team_winrate = pd.read_sql(team_win_rates, conn)
print(df_team_winrate)


                           team  total_wins  total_matches  win_rate
0                Gujarat Titans          37             60     61.67
1           Chennai Super Kings         142            251     56.57
2                Mumbai Indians         151            277     54.51
3          Lucknow Super Giants          30             58     51.72
4         Kolkata Knight Riders         135            264     51.14
5       Rising Pune Supergiants          15             30     50.00
6   Royal Challengers Bengaluru         132            270     48.89
7              Rajasthan Royals         114            234     48.72
8           Sunrisers Hyderabad         122            270     45.19
9                  Punjab Kings         119            264     45.08
10               Delhi Capitals         118            266     44.36
11                Gujarat Lions          13             30     43.33
12         Kochi Tuskers Kerala           6             14     42.86
13                Pune Warriors   

In [23]:
top_wicket_taker = """
SELECT bowler,
       total_wickets,
       RANK() OVER (ORDER BY total_wickets DESC) as rank
FROM (
    SELECT bowler,
           COUNT(*) as total_wickets
    FROM ipl
    WHERE wicket_kind NOT IN 
          ('none', 'run out', 'obstructing the field')
    GROUP BY bowler
)
ORDER BY total_wickets DESC
LIMIT 10
"""

df_wicket_takers = pd.read_sql(top_wicket_taker, conn)
print(df_wicket_takers)

       bowler  total_wickets  rank
0   YS Chahal            221     1
1     B Kumar            198     2
2   PP Chawla            192     3
3   SP Narine            192     3
4    R Ashwin            188     5
5   JJ Bumrah            186     6
6    DJ Bravo            183     7
7    A Mishra            174     8
8   RA Jadeja            170     9
9  SL Malinga            170     9


In [21]:
best_batsman_by_phase = """
SELECT phase,
       batter,
       phase_runs,
       rank
FROM (
    SELECT phase,
           batter,
           SUM(runs_batter) as phase_runs,
           RANK() OVER (PARTITION BY phase 
                        ORDER BY SUM(runs_batter) DESC) as rank
    FROM ipl
    GROUP BY phase, batter
)
WHERE rank <= 3
ORDER BY phase, rank
"""

df_phase_analysis = pd.read_sql(best_batsman_by_phase, conn)
print(df_phase_analysis)

       phase      batter  phase_runs  rank
0      Death    MS Dhoni        2936     1
1      Death  KA Pollard        1708     2
2      Death  KD Karthik        1565     3
3     Middle     V Kohli        4015     1
4     Middle   RG Sharma        3178     2
5     Middle    SK Raina        2999     3
6  Powerplay    S Dhawan        3776     1
7  Powerplay   DA Warner        3648     2
8  Powerplay     V Kohli        3535     3


In [22]:
season_runs_trend = """
SELECT season,
       SUM(runs_batter) as total_runs,
       COUNT(DISTINCT match_id) as total_matches,
       ROUND(SUM(runs_batter) * 1.0 / 
             COUNT(DISTINCT match_id), 2) as runs_per_match
FROM ipl
GROUP BY season
ORDER BY season
"""

df_season_trend = pd.read_sql(season_runs_trend, conn)
print(df_season_trend)

    season  total_runs  total_matches  runs_per_match
0     2007       16809             58          289.81
1     2009       33130            117          283.16
2     2011       19928             73          272.99
3     2012       21323             74          288.15
4     2013       21487             76          282.72
5     2014       17943             60          299.05
6     2015       17427             59          295.37
7     2016       17962             60          299.37
8     2017       17920             59          303.73
9     2018       19098             60          318.30
10    2019       18607             60          310.12
11    2020       18566             60          309.43
12    2021       17727             60          295.45
13    2022       23052             74          311.51
14    2023       24428             74          330.11
15    2024       24657             71          347.28
16    2025       25309             74          342.01
